In [41]:
import pandas as pd
import numpy as np 
import yfinance as yf
import math
from scipy import stats

In [42]:
tickers = pd.read_excel('Book1.xlsx')
tickers.head(5)

,Ticker,Company name
0,ADANIENT,Adani Enterprises Ltd.
1,ADANIPORTS,Adani Ports and Special Economic Zone Ltd.
2,APOLLOHOSP,Apollo Hospitals Enterprise Ltd.
3,ASIANPAINT,Asian Paints Ltd.
4,AXISBANK,Axis Bank Ltd.


In [43]:
def fetch_values_of_stocks(tickers):
    value_cols = [
        "Ticker",
        "Price",
        "PE-Ratio",
        "PB-Ratio",
        "PS-Ratio",
        "EV/EBITDA",
        "EV/GP",
    ]

    rows = []

    def to_number(value):
        try:
            number = float(value)
            return number if np.isfinite(number) else np.nan
        except (TypeError, ValueError):
            return np.nan

    for ticker in tickers:
        if pd.isna(ticker):
            continue

        ticker = str(ticker).strip().upper()
        if not ticker:
            continue
        yahoo_symbol = ticker if ticker.endswith((".NS", ".BO")) else f"{ticker}.NS"
        stock = yf.Ticker(yahoo_symbol)
        price = np.nan
        try:
            history = stock.history(period="5d")
            if not history.empty and "Close" in history.columns:
                close_prices = history["Close"].dropna()
                if not close_prices.empty:
                    price = to_number(close_prices.iloc[-1])
        except Exception as exc:
            print(f"Warning: could not fetch price for {yahoo_symbol}: {exc}")
        try:
            info = stock.info or {}
        except Exception as exc:
            print(f"Warning: could not fetch financial info for {yahoo_symbol}: {exc}")
            info = {}
        pe_ratio = to_number(info.get("forwardPE"))
        if pd.isna(pe_ratio):
            pe_ratio = to_number(info.get("trailingPE"))

        pb_ratio = to_number(info.get("priceToBook"))
        ps_ratio = to_number(info.get("priceToSalesTrailing12Months"))
        enterprise_value = to_number(info.get("enterpriseValue"))
        ebitda = to_number(info.get("ebitda"))
        gross_margin = to_number(info.get("grossMargins"))
        total_revenue = to_number(info.get("totalRevenue"))

        ev_ebitda = (
            enterprise_value / ebitda
            if pd.notna(enterprise_value) and pd.notna(ebitda) and ebitda != 0
            else np.nan
        )

        gross_profit = (
            gross_margin * total_revenue
            if pd.notna(gross_margin) and pd.notna(total_revenue)
            else np.nan
        )
        ev_gross_profit = (
            enterprise_value / gross_profit
            if pd.notna(enterprise_value) and pd.notna(gross_profit) and gross_profit != 0
            else np.nan
        )

        rows.append({
            "Ticker": ticker,
            "Price": price,
            "PE-Ratio": pe_ratio,
            "PB-Ratio": pb_ratio,
            "PS-Ratio": ps_ratio,
            "EV/EBITDA": ev_ebitda,
            "EV/GP": ev_gross_profit,
        })

    return pd.DataFrame(rows, columns=value_cols)


In [44]:
tickers_list = tickers["Ticker"].dropna().astype(str).str.strip().tolist()
df = fetch_values_of_stocks(tickers_list)
df.head(50)

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA,EV/GP
0,ADANIENT,2490.000000,44.358673,3.977439,3.024973,28.549346,8.847385
1,ADANIPORTS,1754.500000,22.266660,4.212535,9.998153,19.892640,17.795778
2,APOLLOHOSP,7791.500000,35.780315,11.817371,4.238753,32.256951,12.806271
3,ASIANPAINT,2277.000000,37.591860,10.219606,5.868793,31.124181,13.242665
4,AXISBANK,1205.699951,10.302855,1.622947,4.640302,NaN,NaN
5,BSE,3441.100098,38.627560,17.722190,24.143478,28.914157,23.413043
6,BAJAJ-AUTO,10390.000000,21.713984,7.478328,3.945746,18.205371,12.274422
7,BAJFINANCE,912.000000,18.166124,4.607131,11.447031,NaN,21.662588
8,BAJAJFINSV,1677.000000,17.699210,3.441045,1.711915,11.636460,8.470165
9,BEL,367.950012,32.786230,11.212176,9.365883,31.810476,18.958568


In [45]:
value_cols = [
        "PE-Ratio",
        "PB-Ratio",
        "PS-Ratio",
        "EV/EBITDA",
        "EV/GP",
    ]
for col in value_cols:
 df[col] = df[col].fillna(df[col].mean())

 df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ticker     50 non-null     object 
 1   Price      50 non-null     float64
 2   PE-Ratio   50 non-null     float64
 3   PB-Ratio   50 non-null     float64
 4   PS-Ratio   50 non-null     float64
 5   EV/EBITDA  42 non-null     float64
 6   EV/GP      45 non-null     float64
dtypes: float64(6), object(1)
memory usage: 2.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ticker     50 non-null     object 
 1   Price      50 non-null     float64
 2   PE-Ratio   50 non-null     float64
 3   PB-Ratio   50 non-null     float64
 4   PS-Ratio   50 non-null     float64
 5   EV/EBITDA  42 non-null     float64
 6   EV/GP      45 non-null     float64
dtypes: float64(6), obje

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ticker     50 non-null     object 
 1   Price      50 non-null     float64
 2   PE-Ratio   50 non-null     float64
 3   PB-Ratio   50 non-null     float64
 4   PS-Ratio   50 non-null     float64
 5   EV/EBITDA  50 non-null     float64
 6   EV/GP      50 non-null     float64
dtypes: float64(6), object(1)
memory usage: 2.9+ KB


In [47]:
Percentile_metrics = {
    "PE-Ratio":"PE-Ratio_percentile",
    "PB-Ratio":"PB-Ratio_percentile",
    "PS-Ratio":"PS-Ratio_percentile",
    "EV/EBITDA":"EV/EBITDA_percentile",
    "EV/GP":"EV/GP_percentile"
}


for metrics, percentile in Percentile_metrics.items():
    df[percentile] = df[metrics].apply(lambda x: stats.percentileofscore(df[metrics], x)/100)

df.head()


,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA,EV/GP,PE-Ratio_percentile,PB-Ratio_percentile,PS-Ratio_percentile,EV/EBITDA_percentile,EV/GP_percentile
0,ADANIENT,2490.000000,44.358673,3.977439,3.024973,28.549346,8.847385,0.88,0.46,0.38,0.54,0.46
1,ADANIPORTS,1754.500000,22.266660,4.212535,9.998153,19.892640,17.795778,0.62,0.48,0.90,0.46,0.74
2,APOLLOHOSP,7791.500000,35.780315,11.817371,4.238753,32.256951,12.806271,0.76,0.90,0.56,0.62,0.60
3,ASIANPAINT,2277.000000,37.591860,10.219606,5.868793,31.124181,13.242665,0.80,0.86,0.70,0.58,0.62
4,AXISBANK,1205.699951,10.302855,1.622947,4.640302,82.985989,44.395771,0.14,0.14,0.62,0.87,0.92


In [48]:
from statistics import mean

df['Value_Score'] = df[[value for value in Percentile_metrics.values()]].mean(axis=1) 

df

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA,EV/GP,PE-Ratio_percentile,PB-Ratio_percentile,PS-Ratio_percentile,EV/EBITDA_percentile,EV/GP_percentile,Value_Score
0,ADANIENT,2490.000000,44.358673,3.977439,3.024973,28.549346,8.847385,0.88,0.46,0.38,0.54,0.46,0.544
1,ADANIPORTS,1754.500000,22.266660,4.212535,9.998153,19.892640,17.795778,0.62,0.48,0.90,0.46,0.74,0.640
2,APOLLOHOSP,7791.500000,35.780315,11.817371,4.238753,32.256951,12.806271,0.76,0.90,0.56,0.62,0.60,0.688
3,ASIANPAINT,2277.000000,37.591860,10.219606,5.868793,31.124181,13.242665,0.80,0.86,0.70,0.58,0.62,0.712
4,AXISBANK,1205.699951,10.302855,1.622947,4.640302,82.985989,44.395771,0.14,0.14,0.62,0.87,0.92,0.538
5,BSE,3441.100098,38.627560,17.722190,24.143478,28.914157,23.413043,0.82,0.92,0.94,0.56,0.84,0.816
6,BAJAJ-AUTO,10390.000000,21.713984,7.478328,3.945746,18.205371,12.274422,0.58,0.70,0.52,0.40,0.56,0.552
7,BAJFINANCE,912.000000,18.166124,4.607131,11.447031,82.985989,21.662588,0.42,0.56,0.92,0.87,0.80,0.714
8,BAJAJFINSV,1677.000000,17.699210,3.441045,1.711915,11.636460,8.470165,0.36,0.44,0.28,0.22,0.44,0.348
9,BEL,367.950012,32.786230,11.212176,9.365883,31.810476,18.958568,0.72,0.88,0.86,0.60,0.78,0.768


In [49]:
df = df.sort_values(by='Value_Score', ascending=False)

df

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA,EV/GP,PE-Ratio_percentile,PB-Ratio_percentile,PS-Ratio_percentile,EV/EBITDA_percentile,EV/GP_percentile,Value_Score
48,TRENT,2820.600098,55.958980,21.533270,7.180728,51.264117,16.890333,0.98,0.94,0.78,0.76,0.72,0.836
34,NESTLEIND,1180.500000,47.583786,39.371000,8.913064,37.805524,15.884613,0.94,1.00,0.84,0.68,0.70,0.832
5,BSE,3441.100098,38.627560,17.722190,24.143478,28.914157,23.413043,0.82,0.92,0.94,0.56,0.84,0.816
47,TITAN,4365.899902,48.886944,24.683113,4.191780,46.608202,21.897803,0.96,0.96,0.54,0.74,0.82,0.804
17,HCLTECH,1420.000000,17.727537,5.131649,258.832800,1372.000748,759.351798,0.38,0.64,1.00,1.00,0.98,0.800
9,BEL,367.950012,32.786230,11.212176,9.365883,31.810476,18.958568,0.72,0.88,0.86,0.60,0.78,0.768
24,INFY,1163.000000,14.553993,4.833904,231.285310,1043.337536,776.821989,0.26,0.60,0.98,0.98,1.00,0.764
15,ETERNAL,331.000000,106.058815,9.449583,3.911730,218.263336,11.177380,1.00,0.84,0.50,0.96,0.52,0.764
32,MAXHEALTH,845.000000,36.182053,7.651699,9.440340,37.871205,15.615244,0.78,0.76,0.88,0.70,0.68,0.760
27,JIOFIN,207.240005,44.792477,1.067466,34.217037,82.985989,41.828427,0.90,0.06,0.96,0.87,0.86,0.730


In [50]:
df.head(10)

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA,EV/GP,PE-Ratio_percentile,PB-Ratio_percentile,PS-Ratio_percentile,EV/EBITDA_percentile,EV/GP_percentile,Value_Score
48,TRENT,2820.600098,55.958980,21.533270,7.180728,51.264117,16.890333,0.98,0.94,0.78,0.76,0.72,0.836
34,NESTLEIND,1180.500000,47.583786,39.371000,8.913064,37.805524,15.884613,0.94,1.00,0.84,0.68,0.70,0.832
5,BSE,3441.100098,38.627560,17.722190,24.143478,28.914157,23.413043,0.82,0.92,0.94,0.56,0.84,0.816
47,TITAN,4365.899902,48.886944,24.683113,4.191780,46.608202,21.897803,0.96,0.96,0.54,0.74,0.82,0.804
17,HCLTECH,1420.000000,17.727537,5.131649,258.832800,1372.000748,759.351798,0.38,0.64,1.00,1.00,0.98,0.800
9,BEL,367.950012,32.786230,11.212176,9.365883,31.810476,18.958568,0.72,0.88,0.86,0.60,0.78,0.768
24,INFY,1163.000000,14.553993,4.833904,231.285310,1043.337536,776.821989,0.26,0.60,0.98,0.98,1.00,0.764
15,ETERNAL,331.000000,106.058815,9.449583,3.911730,218.263336,11.177380,1.00,0.84,0.50,0.96,0.52,0.764
32,MAXHEALTH,845.000000,36.182053,7.651699,9.440340,37.871205,15.615244,0.78,0.76,0.88,0.70,0.68,0.760
27,JIOFIN,207.240005,44.792477,1.067466,34.217037,82.985989,41.828427,0.90,0.06,0.96,0.87,0.86,0.730
